# Sentinel-2 monthly composites for Kakamega (2020-2023, 2025-2026)
Downloads straight to `F:\s2_bands`, with no Google Drive involved.

**Method**
1. Drop scenes with `CLOUDY_PIXEL_PERCENTAGE > 60`, then mask remaining cloudy pixels with Cloud Score+.
2. Build a monthly median composite.
3. Fill gaps from nearby dates: first a ±1 month window, then ±2 months.
4. Add a `fill_flag` band: `0` = original month, `1` = filled from ±1 month, `2` = filled from ±2 months, `255` = no valid data (or outside the county). Spectral bands are `0` where there is no data.
5. Clip to each county and download it tile by tile (Earth Engine limits each direct download to a small size), stitching the tiles into **one GeoTIFF per month** with 11 bands: B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12, fill_flag.

**Notes**
- The download is resumable: finished files are skipped, and a half-written file is never kept (it is written as `.part` and renamed when complete). If it stops, just run the download cell again.
- At 10 m this is a lot of requests and can take many hours (the progress bars show an ETA). Setting `SCALE = 20` makes it roughly 4 times faster and smaller.

In [1]:
# Run once
%pip install -q earthengine-api geemap geopandas shapely rasterio rioxarray xarray dask tqdm requests numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import ee, geemap, geopandas as gpd

ee.Authenticate()   # first time only; opens a browser login
# The high-volume endpoint is meant for many small requests like the tile downloads below
ee.Initialize(project="lope-512513")

## 1. Settings

In [2]:
COUNTY_SHP = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"   # national admin-1 shapefile
COUNTY_NAME_COL = "adm1_name"
COUNTIES = ["Kakamega"]

YEARS = [2020, 2021, 2022, 2023, 2025, 2026]   # 2024 is skipped
LAST_MONTH_2026 = 9                            # 2026 is only downloaded up to this month

MAX_CLOUD = 60            # scene-level cloud cover limit (%)
CS_THRESHOLD = 0.60       # Cloud Score+ pixel clarity (higher = stricter masking)
FILL_WINDOWS = [1, 2]     # months either side used for gap filling
SCALE = 10                # metres; 20 is about 4x faster and smaller

LOCAL_DIR = r"F:\s2_bands"   # where the GeoTIFFs are saved
TILE_PX = 1024               # tile size in pixels (1024 x 1024 x 11 bands x 2 bytes = ~23 MB, under the 48 MB limit)
N_THREADS = 6                # parallel tile downloads; lower this if you see "too many requests" errors

BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

# UTM zone for Kakamega
EXPORT_CRS = {"Kakamega": "EPSG:32636"}

## 2. Load counties from the shapefile

In [3]:
gdf = gpd.read_file(COUNTY_SHP).to_crs(4326)
gdf = gdf[gdf[COUNTY_NAME_COL].str.lower().isin([c.lower() for c in COUNTIES])].copy()
print("Matched counties:", gdf[COUNTY_NAME_COL].tolist())
assert len(gdf) == len(COUNTIES), "Not all counties matched - check names in the shapefile"

# Standardise names to the ones in COUNTIES
name_map = {c.lower(): c for c in COUNTIES}
gdf[COUNTY_NAME_COL] = gdf[COUNTY_NAME_COL].str.lower().map(name_map)

county_geoms = {}      # Earth Engine geometries
county_utm = {}        # shapely geometries in each county's export CRS (used to plan tiles)
for name in COUNTIES:
    sub = gdf[gdf[COUNTY_NAME_COL] == name]
    county_geoms[name] = geemap.gdf_to_ee(sub).geometry()
    g = sub.to_crs(EXPORT_CRS[name]).geometry
    county_utm[name] = g.union_all() if hasattr(g, "union_all") else g.unary_union

all_aoi = geemap.gdf_to_ee(gdf).geometry()

Matched counties: ['Kakamega']


C:\Users\locateIT1\anaconda3\envs\pest-disease-modelling\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


## 3. Sentinel-2 collection, cloud filtering and masking

In [4]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(all_aoi)
      .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD)))

cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2 = s2.linkCollection(cs_plus, ["cs_cdf"])

def mask_clouds(img):
    return img.updateMask(img.select("cs_cdf").gte(CS_THRESHOLD)).select(BANDS)

s2 = s2.map(mask_clouds)

## 4. Monthly composite with gap filling from nearby months

In [5]:
EMPTY = ee.Image.constant([0] * len(BANDS)).rename(BANDS).toUint16().updateMask(0)

def composite(start, end):
    col = s2.filterDate(start, end)
    # if a window has no scenes at all, return a fully masked image instead of failing
    return ee.Image(ee.Algorithms.If(col.size().gt(0), col.median(), EMPTY))

def build_month(y, m):
    start = ee.Date.fromYMD(y, m, 1)
    base = composite(start, start.advance(1, "month"))

    filled = base
    # 255 = still empty, 0 = original month, w = filled from +/- w months
    flag = ee.Image(255).where(base.select("B2").mask(), 0)

    for w in FILL_WINDOWS:
        win = composite(start.advance(-w, "month"), start.advance(w + 1, "month"))
        newly = flag.eq(255).And(win.select("B2").mask())
        flag = flag.where(newly, w)
        filled = filled.unmask(win)

    return (filled.unmask(0).toInt16()
            .addBands(flag.rename("fill_flag").toInt16()))

def county_image(img, geom):
    """Clip to the county; outside it the spectral bands are 0 and fill_flag is 255."""
    clipped = img.clip(geom)
    return (clipped.select(BANDS).unmask(0, False)
            .addBands(clipped.select("fill_flag").unmask(255, False))).toInt16()

## 6. Download functions
Each county-month is split into tiles, downloaded in parallel with retries, and written into one GeoTIFF (compressed). Tiles that lie entirely outside the county are not downloaded.

In [6]:
import os, io, math, time, random, zipfile, shutil
import numpy as np
import requests
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import Affine
from rasterio.windows import Window
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import box
from tqdm.auto import tqdm

OUT_BANDS = BANDS + ["fill_flag"]
NB = len(OUT_BANDS)
os.makedirs(LOCAL_DIR, exist_ok=True)

def out_path(name, y, m):
    return os.path.join(LOCAL_DIR, f"S2_{name}_{y}_{m:02d}.tif")

def fetch_tile(image, crs, x0, y_top, scale, px, tries=6):
    """Download one px x px tile whose top-left corner is (x0, y_top) in `crs`. Returns the raw response bytes."""
    region = ee.Geometry.Rectangle([x0, y_top - px * scale, x0 + px * scale, y_top], proj=crs, geodesic=False)
    params = {"region": region, "crs": crs,
              "crs_transform": [scale, 0, x0, 0, -scale, y_top],
              "format": "GEO_TIFF"}
    last = None
    for i in range(tries):
        try:
            url = image.getDownloadURL(params)
            r = requests.get(url, timeout=600)
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
            return r.content
        except Exception as e:
            last = e
            time.sleep(min(60, 2 ** i + random.random()))
    raise RuntimeError(f"tile failed after {tries} tries: {last}")

def decode_tile(content):
    """Return (array[bands, rows, cols], affine transform) from a downloaded tile."""
    if content[:2] == b"PK":          # zipped, one tif per band
        zf = zipfile.ZipFile(io.BytesIO(content))
        arrs, tr = [], None
        for b in OUT_BANDS:
            member = [n for n in zf.namelist() if n.endswith(f".{b}.tif")][0]
            with MemoryFile(zf.read(member)) as mf, mf.open() as src:
                arrs.append(src.read(1))
                tr = src.transform
        arr = np.stack(arrs)
    else:
        with MemoryFile(content) as mf, mf.open() as src:
            arr, tr = src.read(), src.transform
    if arr.shape[0] != NB:
        raise RuntimeError(f"expected {NB} bands, got {arr.shape[0]}")
    return arr, tr

def download_county_month(name, y, m, img=None):
    final = out_path(name, y, m)
    if os.path.exists(final):
        return "exists"

    crs = EXPORT_CRS[name]
    geom_utm = county_utm[name]
    minx, miny, maxx, maxy = geom_utm.bounds
    X0 = math.floor(minx / SCALE) * SCALE            # top-left corner of the raster grid
    Y0 = math.ceil(maxy / SCALE) * SCALE
    W = math.ceil((maxx - X0) / SCALE)
    H = math.ceil((Y0 - miny) / SCALE)

    image = county_image(img if img is not None else build_month(y, m), county_geoms[name])

    tiles, skipped = [], []
    for r0 in range(0, H, TILE_PX):
        for c0 in range(0, W, TILE_PX):
            tb = box(X0 + c0 * SCALE, Y0 - (r0 + TILE_PX) * SCALE,
                     X0 + (c0 + TILE_PX) * SCALE, Y0 - r0 * SCALE)
            (tiles if tb.intersects(geom_utm) else skipped).append((r0, c0))

    part = final + ".part"
    profile = dict(driver="GTiff", height=H, width=W, count=NB, dtype="int16", crs=crs,
                   transform=Affine(SCALE, 0, X0, 0, -SCALE, Y0),
                   compress="deflate", predictor=2, tiled=True, blockxsize=256, blockysize=256,
                   BIGTIFF="IF_SAFER")
    try:
        with rasterio.open(part, "w", **profile) as dst:
            for i, b in enumerate(OUT_BANDS, 1):
                dst.set_band_description(i, b)
            # tiles fully outside the county: flag = 255 (no data), spectral bands stay 0
            for (r0, c0) in skipped:
                h, w = min(TILE_PX, H - r0), min(TILE_PX, W - c0)
                dst.write(np.full((h, w), 255, dtype="int16"), NB, window=Window(c0, r0, w, h))

            with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
                futs = {ex.submit(fetch_tile, image, crs, X0 + c0 * SCALE, Y0 - r0 * SCALE, SCALE, TILE_PX): (r0, c0)
                        for (r0, c0) in tiles}
                try:
                    for fut in tqdm(as_completed(futs), total=len(futs), leave=False, unit="tile",
                                    desc=f"{name} {y}-{m:02d}"):
                        arr, tr = decode_tile(fut.result())
                        col = round((tr.c - X0) / SCALE)       # place by the tile's own geotransform
                        row = round((Y0 - tr.f) / SCALE)
                        h, w = min(arr.shape[1], H - row), min(arr.shape[2], W - col)
                        dst.write(arr[:, :h, :w].astype("int16"), window=Window(col, row, w, h))
                except BaseException:
                    ex.shutdown(wait=False, cancel_futures=True)
                    raise
        os.replace(part, final)
    except BaseException:
        try:
            if os.path.exists(part):
                os.remove(part)
        except OSError:
            pass
        raise
    return "downloaded"

## 8. Download everything
Two progress bars: the overall bar counts finished files (69 months for this county, with an ETA), and a second bar under it counts tiles for the file in progress.
Already-finished files are skipped. If you stop the cell or something fails, run it again to resume. Failed files are listed at the end.

In [ ]:
months = [(y, m) for y in YEARS for m in range(1, 13)
          if not (y == 2026 and m > LAST_MONTH_2026)]
jobs = [(name, y, m) for (y, m) in months for name in COUNTIES]
already = sum(os.path.exists(out_path(*j)) for j in jobs)
print(f"{len(jobs)} files in total, {already} already on disk")

failed = []
with tqdm(total=len(jobs), initial=already, desc="Files", unit="file") as pbar:
    for (y, m) in months:
        month_img = None
        for name in COUNTIES:
            if os.path.exists(out_path(name, y, m)):
                continue
            pbar.set_postfix_str(f"{name} {y}-{m:02d}")
            try:
                if month_img is None:
                    month_img = build_month(y, m)
                download_county_month(name, y, m, month_img)
            except Exception as e:
                failed.append((name, y, m, str(e)[:200]))
                tqdm.write(f"FAILED {name} {y}-{m:02d}: {str(e)[:200]}")
            pbar.update(1)

print("Done." if not failed else f"Done with {len(failed)} failed file(s); run this cell again to retry them.")
for f_ in failed:
    print("  ", f_)

69 files in total, 5 already on disk


Files:   7%|7         | 5/69 [00:00<?, ?file/s]

Kakamega 2020-06:   0%|          | 0/47 [00:01<?, ?tile/s]

Kakamega 2020-07:   0%|          | 0/47 [00:01<?, ?tile/s]

Kakamega 2020-08:   0%|          | 0/47 [00:00<?, ?tile/s]

Kakamega 2020-09:   0%|          | 0/47 [00:00<?, ?tile/s]

... (30 outputs left out)

Kakamega 2022-12:   0%|          | 0/47 [00:01<?, ?tile/s]

Kakamega 2023-01:   0%|          | 0/47 [00:01<?, ?tile/s]

FAILED Kakamega 2023-01: tile failed after 6 tries: HTTP 429: {
  "error": {
    "code": 429,
    "message": "Too Many Requests: Exceeded Earth Engine concurrency limit. Your project is in Restricted Mode. Learn more at https


Kakamega 2023-02:   0%|          | 0/47 [00:01<?, ?tile/s]

## 9. (Optional) Stack the downloaded months for the county
Builds a lazy stack for the county with a `time` dimension and named bands from the files in `F:\s2_bands`. Nothing is loaded into memory until you compute, plot or save.

In [ ]:
import glob, re
import pandas as pd
import xarray as xr
import rioxarray

stacks = {}
for name in COUNTIES:
    files = sorted(glob.glob(os.path.join(LOCAL_DIR, f"S2_{name}_*.tif")))   # names sort chronologically
    if not files:
        print(f"No files found for {name}")
        continue

    times = [pd.Timestamp(int(y), int(m), 1)
             for y, m in (re.search(r"_(\d{4})_(\d{2})\.tif$", f).groups() for f in files)]

    das = [rioxarray.open_rasterio(f, chunks={"x": 1024, "y": 1024}) for f in files]
    da = xr.concat(das, dim=pd.Index(times, name="time"))
    da = da.assign_coords(band=OUT_BANDS)
    da.name = "S2"
    stacks[name] = da
    print(name, dict(da.sizes))

# Example (cast to float before band maths; the data are int16):
# da = stacks["Kakamega"].astype("float32")
# valid = da.sel(band="fill_flag") < 255
# ndvi = ((da.sel(band="B8") - da.sel(band="B4")) / (da.sel(band="B8") + da.sel(band="B4"))).where(valid)